# Create Gruber Science Prizes Awards (PRIZE PATTERN)

Gruber Science Prize laureates from the official Yale-hosted Gruber Foundation pages.

**Awarding body:** Gruber Foundation - F4320312392, DOI 10.13039/100010848, no ROR in OpenAlex.

**Scope:** Cosmology, Genetics, and Neuroscience prize rows only. The official `/year` page also contains archived Justice and Women's Rights prizes; those are excluded from this science-prize ingest.

**Prerequisites:**
- Run `scripts/local/gruber_prizes_to_s3.py` first.
- Admin must upload the generated parquet because contractor access has no S3 credentials.

**Data sources:**
- `https://gruber.yale.edu/year` for official laureate rows
- `https://gruber.yale.edu/cosmology`, `https://gruber.yale.edu/genetics`, and `https://gruber.yale.edu/neuroscience` for official USD 500,000 amount rule
- Per-prize pages under `https://gruber.yale.edu/prize/...`
- Per-recipient profiles under `https://gruber.yale.edu/recipient/...`

**S3:** `s3a://openalex-ingest/awards/gruber_prizes/gruber_prizes_awards.parquet`

**Mapping notes:**
- Prize pattern: one row per `(science prize x laureate)`.
- `lead_investigator` is the official laureate name from the Gruber page.
- Some official laureates are teams; for those rows `given_name` is NULL and `family_name` carries the full official team string.
- Structured affiliation/country is not published consistently by the official source; leave affiliation fields NULL rather than backfilling from Wikipedia/Wikidata.
- Official yearly/category prize total is USD 500,000. Amount is split by the official laureate count for that year/category so shared prizes do not multiply the total.
- Provenance is `gruber_prizes`.
- Priority is 85; priorities 66-84 are reserved by in-flight fork PRs, including The Brain Prize at 84.


## Step 1: Create staging table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gruber_prizes_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gruber_prizes/gruber_prizes_awards.parquet`;


## Step 1.5: Inspect raw data before transform


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows FROM openalex.awards.gruber_prizes_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.gruber_prizes_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.gruber_prizes_raw LIMIT 10;


In [ ]:
%sql
-- Required money-field scan before mapping.
SELECT column_name
FROM (DESCRIBE openalex.awards.gruber_prizes_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|portion|laureate_count';


In [ ]:
%sql
-- Required currency-field scan before mapping.
SELECT column_name
FROM (DESCRIBE openalex.awards.gruber_prizes_raw)
WHERE LOWER(column_name) RLIKE 'currenc|currency|ccy|iso_4217';


In [ ]:
%sql
SELECT
    prize_category,
    COUNT(*) AS rows,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year,
    COUNT(source_teaser) AS rows_with_teaser,
    COUNT(prize_profile_text) AS rows_with_prize_profile,
    COUNT(recipient_profile_url) AS rows_with_recipient_url,
    COUNT(recipient_bio_text) AS rows_with_recipient_bio
FROM openalex.awards.gruber_prizes_raw
GROUP BY prize_category
ORDER BY prize_category;


In [ ]:
%sql
-- Sanity-check the source money columns. Official category pages state
-- USD 500,000; the transform below splits that total by laureate_count.
SELECT
    COUNT(*) AS total_rows,
    COUNT(source_total_award_amount) AS rows_with_total_amount,
    COUNT(source_currency) AS rows_with_currency,
    collect_set(source_currency) AS currencies,
    MIN(TRY_CAST(source_total_award_amount AS DOUBLE)) AS min_total_amount,
    MAX(TRY_CAST(source_total_award_amount AS DOUBLE)) AS max_total_amount,
    AVG(TRY_CAST(source_total_award_amount AS DOUBLE)) AS avg_total_amount,
    MIN(TRY_CAST(laureate_count AS DOUBLE)) AS min_laureate_count,
    MAX(TRY_CAST(laureate_count AS DOUBLE)) AS max_laureate_count
FROM openalex.awards.gruber_prizes_raw;


## Step 1.6: Funder existence fail-fast


In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP and do not run the transform.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320312392;


## Step 2: Transform to award schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gruber_prizes_awards
USING delta
AS
WITH gruber_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320312392
),
normalized AS (
    SELECT
        g.*,
        TRY_CAST(g.award_year AS INT) AS parsed_award_year,
        TRY_CAST(g.source_total_award_amount AS DOUBLE) AS parsed_total_amount,
        TRY_CAST(g.laureate_count AS DOUBLE) AS parsed_laureate_count
    FROM openalex.awards.gruber_prizes_raw g
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':gruber_prizes:', LOWER(TRIM(g.funder_award_id))))) % 9000000000 AS id,
    CONCAT(g.source_title, ' - ', g.laureate_name) AS display_name,
    COALESCE(NULLIF(g.source_teaser, ''), NULLIF(g.prize_profile_text, ''), NULLIF(g.recipient_bio_text, '')) AS description,
    f.funder_id,
    g.funder_award_id AS funder_award_id,
    CASE
        WHEN g.parsed_total_amount IS NOT NULL AND g.parsed_laureate_count > 0
            THEN g.parsed_total_amount / g.parsed_laureate_count
        ELSE NULL
    END AS amount,
    NULLIF(g.source_currency, '') AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    CONCAT('Gruber ', g.prize_category, ' Prize') AS funder_scheme,
    'gruber_prizes' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(g.parsed_award_year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(g.parsed_award_year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    g.parsed_award_year AS start_year,
    g.parsed_award_year AS end_year,
    struct(
        NULLIF(g.laureate_given_name, '') AS given_name,
        NULLIF(g.laureate_family_name, '') AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            CAST(NULL AS STRING) AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    COALESCE(NULLIF(g.recipient_profile_url, ''), NULLIF(g.prize_detail_url, '')) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':gruber_prizes:', LOWER(TRIM(g.funder_award_id))))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized g
CROSS JOIN gruber_funder f
WHERE g.funder_award_id IS NOT NULL
  AND g.laureate_name IS NOT NULL
  AND g.prize_category IN ('Cosmology', 'Genetics', 'Neuroscience')
  AND g.parsed_award_year IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 85


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gruber_prizes' AND priority = 85;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       85 AS priority
FROM openalex.awards.gruber_prizes_awards;


## Verification


In [ ]:
%sql
-- Step 6.7 amount/currency check. Expect 100% USD amount coverage,
-- with values split from the official USD 500,000 category/year total.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.gruber_prizes_awards;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 120) AS display_name,
    funder_scheme,
    amount,
    currency,
    start_year,
    lead_investigator.given_name,
    lead_investigator.family_name,
    landing_page_url
FROM openalex.awards.gruber_prizes_awards
ORDER BY start_year DESC, funder_scheme, lead_investigator.family_name
LIMIT 25;


In [ ]:
%sql
SELECT id, COUNT(*) AS dupes
FROM openalex.awards.gruber_prizes_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS dupes
FROM openalex.awards.gruber_prizes_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year,
    SUM(amount) AS summed_amount
FROM openalex.awards.gruber_prizes_awards
GROUP BY funder_scheme
ORDER BY funder_scheme;


In [ ]:
%sql
-- Data completeness (runbook Step 6.3 form, adapted for prize pattern).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    COUNT(lead_investigator.given_name) AS has_given_name,
    COUNT(lead_investigator.family_name) AS has_family_name,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates
FROM openalex.awards.gruber_prizes_awards;


In [ ]:
%sql
SELECT funder_id, funder.display_name, COUNT(*) AS rows
FROM openalex.awards.gruber_prizes_awards
GROUP BY funder_id, funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.gruber_prizes_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT provenance, priority, COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gruber_prizes' AND priority = 85
GROUP BY provenance, priority;
